In [43]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

In [44]:
spark =(
    SparkSession.builder
    .appName("Sales Fact table")
    .master("local[*]")
    .getOrCreate()
)

In [45]:
# read orders silver

orders_df=spark.read.parquet("output/silver/orders")
orders_df.show(5)
orders_df.printSchema()


+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+-------------------+-------------+--------------+-------------------+------------+----------------+----------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date| purchase timestamp|purchase_year|purchase_month| purchase_timestamp|purchase_day|purchase_quarter|purchase_weekday|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+-------------------+-------------+--------------+-------------------+------------+----------------+----------------+
|0a2407e0197d11314...|d2038fc66997336ea...|   delivered|     2017-10-19 18:06:1

In [46]:
# read order items (bronze)

orders_items_df = spark.read.parquet("output/bronze/order_items")
orders_items_df.show(5)
orders_items_df.printSchema()

+--------------------+-------------+--------------------+--------------------+-------------------+------+-------------+
|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_date| price|freight_value|
+--------------------+-------------+--------------------+--------------------+-------------------+------+-------------+
|8ac26cb701a7887cc...|            1|4ebb87ba41ca44632...|7a67c85e85bb2ce85...|2017-05-22 16:05:14|109.99|        18.02|
|8ac2728285fd4228f...|            1|8b90be4893a4277a9...|004c9cd9d87a3c30c...|2017-03-15 14:09:17|109.99|         8.27|
|8ac2728285fd4228f...|            2|fa94f25a73969e3a2...|004c9cd9d87a3c30c...|2017-03-15 14:09:17|109.99|        16.55|
|8ac2728285fd4228f...|            3|b01cedfa96d891427...|004c9cd9d87a3c30c...|2017-03-15 14:09:17|259.99|        21.01|
|8ac2728285fd4228f...|            4|fa94f25a73969e3a2...|004c9cd9d87a3c30c...|2017-03-15 14:09:17|109.99|        16.55|
+--------------------+-------------+----

In [47]:
# Read Payments (Bronze)

payment_df = spark.read.parquet("output/bronze/payments")
payment_df.show(5)
payment_df.printSchema()

+--------------------+------------------+------------+--------------------+-------------+
|            order_id|payment_sequential|payment_type|payment_installments|payment_value|
+--------------------+------------------+------------+--------------------+-------------+
|b81ef226f3fe1789b...|                 1| credit_card|                   8|        99.33|
|a9810da82917af2d9...|                 1| credit_card|                   1|        24.39|
|25e8ea4e93396b6fa...|                 1| credit_card|                   1|        65.71|
|ba78997921bbcdc13...|                 1| credit_card|                   8|       107.78|
|42fdf880ba16b47b5...|                 1| credit_card|                   2|       128.45|
+--------------------+------------------+------------+--------------------+-------------+
only showing top 5 rows
root
 |-- order_id: string (nullable = true)
 |-- payment_sequential: integer (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_installments:

In [48]:
# Read Products (Bronze)
products_df = spark.read.parquet("output/bronze/products")
products_df.show(5)
products_df.printSchema()

+--------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|          product_id|product_category_name|product_name_lenght|product_description_lenght|product_photos_qty|product_weight_g|product_length_cm|product_height_cm|product_width_cm|
+--------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|1e9e8ef04dbcff454...|           perfumaria|                 40|                       287|                 1|             225|               16|               10|              14|
|3aa071139cb16b67c...|                artes|                 44|                       276|                 1|            1000|               30|               18|              20|
|96bd76ec8810374ed...|        esporte_lazer|                 46|                       250|    

In [49]:
# Show Counts Before Join

print("\nOrders Count:")
print(orders_df.count())

print("\nOrder Items Count:")
print(orders_items_df.count())

print("\nPayments Count:")
print(payment_df.count())

print("\nProducts Count:")
print(products_df.count())



Orders Count:
99441

Order Items Count:
112650

Payments Count:
103886

Products Count:
32951


In [50]:
# Join Orders + Order Items

# since we have order id commaon in both the table orders_df and orders_item_df so based on that we are able to join both the tables usint inner join

sales_df=orders_df.join(
    orders_items_df,    # table which is being joined
    on ="order_id",
    how ="inner"    # specify the type of join you want to perform 
)
print("\nAfter Orders + Items Join")
print(sales_df.count())
sales_df.show(5)


After Orders + Items Join
112650
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+-------------------+-------------+--------------+-------------------+------------+----------------+----------------+-------------+--------------------+--------------------+-------------------+------+-------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date| purchase timestamp|purchase_year|purchase_month| purchase_timestamp|purchase_day|purchase_quarter|purchase_weekday|order_item_id|          product_id|           seller_id|shipping_limit_date| price|freight_value|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+----------------------------

In [51]:
# join payemnt table with sales_df table using order id 

sales_df=sales_df.join(
    payment_df,
    on="order_id",
    how="inner"
)
print("\nAfter Products Join")
print(sales_df.count())
sales_df.show(5)


After Products Join
117601
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+-------------------+-------------+--------------+-------------------+------------+----------------+----------------+-------------+--------------------+--------------------+-------------------+-----+-------------+------------------+------------+--------------------+-------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date| purchase timestamp|purchase_year|purchase_month| purchase_timestamp|purchase_day|purchase_quarter|purchase_weekday|order_item_id|          product_id|           seller_id|shipping_limit_date|price|freight_value|payment_sequential|payment_type|payment_installments|payment_value|
+--------------------+----------

In [ ]:
sales_fact_df = sales_df.select(
    "order_id",
    "customer_id",
    "product_id",
    "seller_id",
    "payment_type",        
    "payment_value",       
    "price",
    "freight_value",
    "purchase_timestamp",
    "purchase_year",
    "purchase_month",
    "purchase_day",
    "purchase_quarter",
    "purchase_weekday"
)

print("\nSales Fact Sample")
sales_fact_df.show(10, truncate=False)


Sales Fact Sample
+--------------------------------+--------------------------------+--------------------------------+--------------------------------+------------+-------------+------+-------------+-------------------+-------------+--------------+------------+----------------+----------------+
|order_id                        |customer_id                     |product_id                      |seller_id                       |payment_type|payment_value|price |freight_value|purchase_timestamp |purchase_year|purchase_month|purchase_day|purchase_quarter|purchase_weekday|
+--------------------------------+--------------------------------+--------------------------------+--------------------------------+------------+-------------+------+-------------+-------------------+-------------+--------------+------------+----------------+----------------+
|0a2407e0197d11314604cf01105f648c|d2038fc66997336ea2fb9a7870fc279c|70b487a87fee25bc56455501bdc83501|da8622b14eb17ae2831f4ac5b9dab84a|credit_card |2

In [55]:
# checking final fact table schema 
sales_fact_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_value: double (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- purchase_timestamp: timestamp (nullable = true)
 |-- purchase_year: integer (nullable = true)
 |-- purchase_month: integer (nullable = true)
 |-- purchase_day: integer (nullable = true)
 |-- purchase_quarter: integer (nullable = true)
 |-- purchase_weekday: string (nullable = true)



In [56]:
# save to gold layer

sales_fact_df.write.mode(
    "overwrite"
).parquet("output/gold/sales_fact")

26/06/12 18:25:34 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/06/12 18:25:34 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/06/12 18:25:34 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/06/12 18:25:35 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/06/12 18:25:35 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
